In [1]:
import json
import re
from pathlib import Path

# Пути
MAPPING_FILE = Path("terms_map.json")   # JSON с заменами
INPUT_DIR = Path("../../parsed_files")              # исходная директория с md-файлами
OUTPUT_DIR = Path("../../knowledge_base_2")     # директория для результатов

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Загружаем словарь замен
with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    mapping = json.load(f)

# --- Расширяем мапу для частичных замен ---
extended_mapping = {}

for key, value in mapping.items():
    extended_mapping[key] = value
    key_parts = key.split()
    value_parts = value.split()
    # Если термины состоят из нескольких слов — мапим их по частям
    if len(key_parts) == len(value_parts) and len(key_parts) > 1:
        for kp, vp in zip(key_parts, value_parts):
            # Добавляем только если части ещё не были добавлены, чтобы не затирать другие замены
            if kp not in extended_mapping:
                extended_mapping[kp] = vp

# Сортируем ключи по длине (чтобы длинные фразы шли первыми)
sorted_terms = sorted(extended_mapping.keys(), key=lambda x: -len(x))

# Регулярка для поиска
pattern = re.compile(
    r"\b(" + "|".join(re.escape(term) for term in sorted_terms) + r")\b",
    flags=re.IGNORECASE
)

def replace_terms(match):
    orig = match.group(0)
    key_lower = orig.lower()
    # ищем замену без учёта регистра
    replacement = None
    for k, v in extended_mapping.items():
        if k.lower() == key_lower:
            replacement = v
            break

    if not replacement:
        return orig

    # сохраняем капитализацию
    if orig.isupper():
        replacement = replacement.upper()
    elif orig[0].isupper():
        replacement = replacement[0].upper() + replacement[1:]
    return replacement

def replace_in_filename(filename: str) -> str:
    """Переименовывает файл согласно extended_mapping"""
    name_stem = Path(filename).stem
    suffix = Path(filename).suffix
    # Замена по тем же правилам
    new_name = pattern.sub(replace_terms, name_stem)
    return new_name + suffix

# --- Обработка файлов ---
for md_file in INPUT_DIR.glob("*.md"):
    print(f"Обрабатываем {md_file.name} ...")
    text = md_file.read_text(encoding="utf-8")
    new_text = pattern.sub(replace_terms, text)

    # Переименовываем файл в соответствии с мапой
    new_filename = replace_in_filename(md_file.name)
    output_file = OUTPUT_DIR / new_filename

    output_file.write_text(new_text, encoding="utf-8")
    print(f"✅ Сохранено: {output_file.name}")

print("Все файлы обработаны.")


Обрабатываем Arges.md ...
✅ Сохранено: Arges.md
Обрабатываем 47 lightsaber nullifier.md ...
✅ Сохранено: 47 Aether Blade nullifier.md
Обрабатываем Corrida.md ...
✅ Сохранено: Corrida.md
Обрабатываем Crystalline reformatter.md ...
✅ Сохранено: Crystalline reformatter.md
Обрабатываем Catalyst reactant cradle.md ...
✅ Сохранено: Catalyst reactant cradle.md
Обрабатываем Conference on Aviles Prime.md ...
✅ Сохранено: Conference on Aviles Prime.md
Обрабатываем Deflector shield.md ...
✅ Сохранено: Deflector shield.md
Обрабатываем Bursant.md ...
✅ Сохранено: Bursant.md
Обрабатываем Brotherhood of the Ninth Door scroll.md ...
✅ Сохранено: Brotherhood of The Ninth Door scroll.md
Обрабатываем Centrifuge.md ...
✅ Сохранено: Centrifuge.md
Обрабатываем Cheater.md ...
✅ Сохранено: Cheater.md
Обрабатываем Clonetroller.md ...
✅ Сохранено: Clonetroller.md
Обрабатываем Chromium timelock.md ...
✅ Сохранено: Chromium timelock.md
Обрабатываем D-93 sonic tri-denax cable.md ...
✅ Сохранено: D-93 sonic tri-den